# compute_beta_files.ipynb
This file takes .spec files (Peter's format), computes beta and signal-to-noise ratios, and outputs .beta* files.




In [1]:
low_f_range = [2, 5]
high_f_range = [20, 35]

spec_dir = "example_data/example_spec_files/"
output_dir = "example_data/example_subset_beta_files/"

# spec_dir = "/Users/ivandevert/projects/ridgecrest2019_prev/proc/spec/SPECOUT/"
# output_dir = "/Users/ivandevert/projects/spectral_falloff_ratio/data/beta_files/"

stn_method = 'time-domain-amplitude'
beta_method = 'beta-method-1'


In [10]:
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import os
import struct
import math
import time
import lib.beta_functions as bf
from lib.beta_functions import read_spec, write_beta_files, read_beta_files

try:
    from geopy import distance
    has_geopy = True
    print("Using geopy package")
except:
    has_geopy = False
    print("Not using geopy package")

try:
    from tqdm import tqdm, trange
except:
    print("Failed to import module tqdm. Please install to show progress bar.")


Using geopy package


### Define functions to calculate STN ratio and beta

In [3]:
# this is so the functions can have a name attribute
def func_name(r):
    def wrapper(f):
        f.name = r 
        return f 
    return wrapper

@func_name("time-domain-amplitude")
def get_stn(spec):
    """ Compute the signal to noise for a single record:
    1) Take derivative of noise and signal windows separately
    2) Demean combined noise/signal window
    3) STN = mean of abs(signal) / mean of abs(noise)

    Args:
        spec (dict): spec object, returned from read_spec()

    Last Modified:
        2024-06-27
    """
    # take derivative to stabilize the demean
    x1 = np.diff(spec['x1'])    # noise window
    x2 = np.diff(spec['x2'])    # signal window

    # demean entire combined window
    mean = np.mean(np.hstack((x1, x2)))
    x1 = x1 - mean 
    x2 = x2 - mean

    # compute STN ratio
    stn = np.mean(np.abs(x2)) / np.mean(np.abs(x1))

    return stn

@ func_name("beta-method-1")
def get_beta(spec, low_f_ind, high_f_ind, df):
    """ Compute the beta parameter:
    1) compute the logmean of a low and high band of the signal spectrum
    2) normalize the mean by the frequency width of each band
    3) beta = high normalized mean / low normalized mean

    Args:
        spec (dict): spec object, returned from read_spec()

    Last Modified:
        2024-06-27
    """
    # assume a linear-scale (non-log) spectrum
    s_low = spec['s2'][low_f_ind[0]:low_f_ind[1]+1]
    s_high = spec['s2'][high_f_ind[0]:high_f_ind[1]+1]

    # log mean value
    low_avg =  np.power(10, np.mean(np.log10(s_low)))  / (df * (low_f_ind[1] - low_f_ind[0]))
    high_avg = np.power(10, np.mean(np.log10(s_high))) / (df * (high_f_ind[1] - high_f_ind[0]))

    beta = high_avg / low_avg
    return beta, low_avg, high_avg

def get_station_name(spec):
    """Helper function to get a unique station name from the data structure
    output by read_spec.

    Args:
        spec (dict): contains various data relating to a spectrum

    Returns:
        stname (str): Station identifier (ex: "CI.ABC..HHZ")

    Sources:

    Last Modified:
        2024-07-22
    """
    stname = ".".join([el.strip() for el in [spec['stype'], spec['stname'], spec['loccode'], spec['chnm']]])
    return stname


### Begin program
First, find all .spec files

In [4]:

# Get a list of all .spec files in spec_dir
spec_event_ids = [int(el.split('.')[0]) for el in os.listdir(spec_dir) \
                    if el.endswith(".spec")]
spec_filepaths = [spec_dir + el for el in os.listdir(spec_dir) \
                    if el.endswith(".spec")]

nevents = len(spec_event_ids)

print(f"{nevents} .spec files found in {spec_dir}")

81 .spec files found in example_data/example_spec_files/


Next, perform checks to make sure we're using the correct functions to calculate $\beta$ and signal-to-noise ratios, and using the correct units for deldist. Also, form the frequency array and determine indices to measure spectral level between.

In [5]:
assert get_stn.name == stn_method, "Check get_stn function names/methods"
assert get_beta.name == beta_method, "Check get_beta function names/methods"

# open a .spec file as a test
shead, ehead, spec = read_spec(spec_filepaths[0])

deldist_avg = 0
for sp in spec:
    deldist_avg += sp['deldist']
deldist_avg /= len(spec)
if deldist_avg <= 10:
    dist_km = False
    print(f"Average deldist is {deldist_avg:.4f}. Interpreting this as degrees.")
elif deldist_avg > 10:
    dist_km = True
    print(f"Average deldist is {deldist_avg:.4f}. Interpreting this as km.")


f = np.arange(shead['nf']) * shead['df']


low_f_ind = np.array([np.argmin(np.abs(f - el)) for el in low_f_range])
high_f_ind = np.array([np.argmin(np.abs(f - el)) for el in high_f_range])

low_f_band = f[low_f_ind]
high_f_band = f[high_f_ind]

print("")
print("FREQUENCY ARRAY INFORMATION")
print(f"Frequency array ranges from {f[0]:.2f} to {f[-1]:.2f} Hz with {len(f)} elements (df = {shead['df']:.3f} Hz). ")
print(f"Desired | Actual low-frequency band:  {low_f_range[0]:7.3f} -{low_f_range[1]:7.3f} Hz | {low_f_band[0]:7.3f} -{low_f_band[1]:7.3f} Hz")
print(f"Desired | Actual high-frequency band: {high_f_range[0]:7.3f} -{high_f_range[1]:7.3f} Hz | {high_f_band[0]:7.3f} -{high_f_band[1]:7.3f} Hz")



Average deldist is 0.5337. Interpreting this as degrees.

FREQUENCY ARRAY INFORMATION
Frequency array ranges from 0.00 to 50.00 Hz with 76 elements (df = 0.667 Hz). 
Desired | Actual low-frequency band:    2.000 -  5.000 Hz |   2.000 -  4.667 Hz
Desired | Actual high-frequency band:  20.000 - 35.000 Hz |  20.000 - 34.667 Hz


### Compute and write .beta files
Each station has one .beta file with all events recorded by that station, while each .spec file represents one event with all stations recording that event. The .beta file is essentially a reorganization of the same data, but with a few added fields (beta and stn, for now).

New way, using explode and groupby

In [11]:
# Each station/row will have a single value for each of (stname, slat, slon,
# selev), while the fields (evids, qmag, beta, stn, deldist) will be variable
# but equal length arrays containing one entry per event.
columns = {
    "stname"  : "object",
    "slat"    : "float", 
    "slon"    : "float", 
    "selev"   : "float", 
    "evids"   : "object", 
    "qmag"    : "object", 
    "qlat"    : "object", 
    "qlon"    : "object", 
    "qdep"    : "object", 
    "beta"    : "object", 
    "stn"     : "object", 
    "deldist" : "object", 
}

# initialize the pandas DataFrame with the above columns and datatypes
df = pd.DataFrame({key: np.empty(dtype=val, shape=0) for key, val in columns.items()})

if 'tqdm' in locals():
    rng = trange(nevents)
else:
    rng = range(nevents)

t0 = time.time()
ii = 0
for i in rng:
    try:
        shead, ehead, spec = read_spec(spec_filepaths[i])

        qlat = ehead['qlat']
        qlon = ehead['qlon']
        qdep = ehead['qdep']
        qmag = ehead['qmag1']

        evid = ehead['cuspid']

        nspec = ehead['numts']

        for j in range(nspec):
            sp = spec[j]
            stname = get_station_name(sp)

            # this is pretty slow
            dist = distance.distance((qlat, qlon), (sp['slat'], sp['slon'])).km

            # beta
            beta, low_avg, high_avg = get_beta(sp, low_f_ind, high_f_ind, shead["df"])

            # signal to noise
            stn = get_stn(sp)

            # if a row for the station exists, append event information to
            # select columns
            if stname in df['stname'].values:
                ind = np.where(df['stname'].values==stname)[0][0]
                df.at[ind, 'evids'] = np.append(df.at[ind, 'evids'], evid)
                df.at[ind, 'qmag'] = np.append(df.at[ind, 'qmag'], qmag)
                df.at[ind, 'qlat'] = np.append(df.at[ind, 'qlat'], qlat)
                df.at[ind, 'qlon'] = np.append(df.at[ind, 'qlon'], qlon)
                df.at[ind, 'qdep'] = np.append(df.at[ind, 'qdep'], qdep)
                df.at[ind, 'beta'] = np.append(df.at[ind, 'beta'], beta)
                df.at[ind, 'stn'] = np.append(df.at[ind, 'stn'], stn)
                df.at[ind, 'deldist'] = np.append(df.at[ind, 'deldist'], dist)


            else:
                # if the station doesn't have a row, append a new row for 
                # the station
                evids = np.array([evid, ])
                qmags = np.array([qmag, ])
                qlats = np.array([qlat, ])
                qlons = np.array([qlon, ])
                qdeps = np.array([qdep, ])
                beta = np.array([beta, ])
                stn = np.array([stn, ])
                deldist = np.array([dist, ])
                df.loc[len(df)] = [stname, sp['slat'], sp['slon'], sp['selev'], evids, qmags, qlats, qlons, qdeps, beta, stn, deldist]
        ii += 1
    except:
        print(f"Error with {spec_filepaths[i]}")

df['beta_method'] = beta_method
df['stn_method'] = stn_method
df['low_f_band'] = [list(low_f_band.astype(float))] * len(df)
df['high_f_band'] = [list(high_f_band.astype(float))] * len(df)
t1 = time.time()

print(f"{t1-t0:.3} s reading and computing from .spec files")
print(f"{(t1-t0)*1000/ii:.2f}ms per .spec file\n")

print("Writing to .beta files...", end="")
t0 = time.time()
write_beta_files(df, output_dir)
t1 = time.time()
print("Done.")
print(f"{t1-t0:.3} s writing .beta files")
print(f"{(t1-t0)*1000/nevents:.2f}ms per .spec file")

100%|██████████| 81/81 [00:15<00:00,  5.35it/s]


15.1 s reading and computing from .spec files
186.95ms per .spec file

Writing to .beta files...Done.
0.568 s writing .beta files
7.02ms per .spec file
